In [14]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from faster_whisper import WhisperModel
from IPython.display import Audio, display
import sounddevice as sd
import soundfile as sf

In [2]:
load_dotenv()
client = OpenAI()

STT engine

In [3]:
# Load the base model
stt_model = WhisperModel(
    'base',
    device='cpu',
    compute_type='int8'
)

print('faster-whisper base loaded')

faster-whisper base loaded


RAG chain

In [4]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings()

vectorstore = Chroma(
    persist_directory=r'C:\Users\USER\rag_course\chroma_db_domain',
    embedding_function=embeddings,
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

rag_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are AgriVoice, a farming assistant for Nigerian farmers. '
     'Answer using ONLY the context below. Keep it under 3 sentences. '
     'If the answer is not in the context, say "I don\'t know".'),
    ('human', 'Context:\n{context}\n\nQuestion: {question}'),
])
rag_chain = rag_prompt | llm | StrOutputParser()

print(f'✅ RAG ready — {vectorstore._collection.count()} chunks')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ RAG ready — 82 chunks


Terminology store

In [5]:
TERMINOLOGY_DIR = r'C:\Users\USER\rag_course\chroma_db_terminology'

terminology_store = Chroma(
    persist_directory=TERMINOLOGY_DIR,
    embedding_function=embeddings,
    collection_name='terminology',
)
terminology_retriever = terminology_store.as_retriever(search_kwargs={'k': 5})

print(f'Terminology ready — {terminology_store._collection.count()} docs')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Terminology ready — 36 docs


Translation helpers

In [6]:
# Retrieve relevant crop/disease terminology from Chroma
def get_relevant_terms(text: str) -> str:
    docs = terminology_retriever.invoke(text)
    
    return '\n'.join(f'- {d.page_content}' for d in docs)

# Translate English text to a Nigerian language
def translate(text: str, target_lang: str) -> str:
    if target_lang == 'English':
        return text
    
    terms = get_relevant_terms(text=text)
    
    system = (
        f'You are an expert translator for Nigerian languages.\n'
        f'Translate the following English text to {target_lang}.\n\n'
        f'MANDATORY VOCABULARY (use these exact words):\n{terms}\n\n'
        f'RULES:\n'
        f'1. Output ONLY {target_lang} — no English mixed in, except proper nouns.\n'
        f'2. Use the correct tone marks and diacritics.\n'
        f'3. Output ONLY the translation. No explanations.\n'
    )
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': text}
        ],
        temperature=0,
    )    
    return response.choices[0].message.content.strip()

# Translate FROM a Nigerian language TO English
def translate_to_english(text: str, source_lang: str) -> str:
    if source_lang == 'English':
        return text
    
    terms = get_relevant_terms(text)
    
    system = (
        f'You are an expert translator for Nigerian languages.\n'
        f'Translate the following {source_lang} text to English.\n\n'
        f'MANDATORY VOCABULARY (use these to decode crops/diseases):\n{terms}\n\n'
        f'RULES:\n'
        f'1. Output ONLY English.\n'
        f'2. Use the terminology above to correctly identify crops.\n'
        f'3. Output ONLY the translation. No explanations.\n'
    )
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': text},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

print('Translation helpers ready')

Translation helpers ready


VoicePipeline 

In [7]:
class VoicePipeline:
    '''One voice turn: audio in → STT → RAG → TTS → audio out.'''

    # Store config once so every method can reuse it without reloading
    def __init__(self, language: str = 'English'):
        self.language = language
        print(f'✅ VoicePipeline created (language={self.language})')

    # Transcribe an audio file to text using local faster-whisper
    def listen(self, audio_path: str) -> str:
        segments, info = stt_model.transcribe(audio_path, beam_size=5)
        return ' '.join(seg.text.strip() for seg in segments)

    # Translate to English → retrieve → answer → translate back to user's language
    def think(self, question: str) -> str:
        question_en = translate_to_english(question, self.language)

        docs = retriever.invoke(question_en)
        context = '\n\n'.join(doc.page_content for doc in docs)
        answer_en = rag_chain.invoke({'context': context, 'question': question_en})

        return translate(answer_en, self.language)

    # Convert the answer text into spoken audio bytes with OpenAI TTS
    def speak(self, answer: str) -> bytes:
        response = client.audio.speech.create(
            model='tts-1',
            voice='alloy',
            input=answer,
        )
        return response.content

    # Chain listen → think → speak — the full one-turn voice flow
    def turn(self, audio_path: str) -> bytes:
        print('🎧 Listening...')
        question = self.listen(audio_path)
        print(f'   📝 {question}')

        print('🧠 Thinking...')
        answer = self.think(question)
        print(f'   💬 {answer}')

        print('🔊 Speaking...')
        audio_bytes = self.speak(answer)

        return audio_bytes


print('✅ VoicePipeline class defined')

✅ VoicePipeline class defined


test the whole pipeline

In [8]:
vp = VoicePipeline(language='English')

audio_in = r'C:\Users\USER\rag_course\12b_voice_agents\data\sample_question.mp3'
audio_bytes = vp.turn(audio_in)

out = r'C:\Users\USER\rag_course\12b_voice_agents\data\turn_fw.mp3'
with open(out, 'wb') as f:
    f.write(audio_bytes)
print(f'💾 {out}')

✅ VoicePipeline created (language=English)
🎧 Listening...
   📝 How do I treat cassava mosaic disease?
🧠 Thinking...


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


   💬 To treat cassava mosaic disease, use disease-free cuttings for planting, select resistant cassava varieties, and immediately remove and destroy infected plants. Control whitefly vectors with appropriate insecticides and avoid planting cassava near infected fields.
🔊 Speaking...
💾 C:\Users\USER\rag_course\12b_voice_agents\data\turn_fw.mp3


In [11]:
display(Audio(out))

Transcribe with faster-whisper

In [13]:
# audio_in = r'C:\Users\USER\rag_course\12b_voice_agents\data\sample_question.mp3'

# # Transcribe — returns a generator of segments + info about the language
# segments, info = stt_model.transcribe(audio_in, beam_size=5)

# print(f'Language: {info.language} (probability {info.language_probability:.2f})')

# print(f'Duration: {info.duration:.2f} seconds')
# print()
# print('Text:')

# for seg in segments:
#     print(f'    [{seg.start:.2f}s -> {seg.end:.2f}] {seg.text}')

Live microphone loop

Test the mic

In [15]:
print('Microphones found:')
print(sd.query_devices())

Microphones found:
   0 Microsoft Sound Mapper - Input, MME (2 in, 0 out)
>  1 Microphone (2- Realtek High Def, MME (2 in, 0 out)
   2 Stereo Mix (2- Realtek High Def, MME (2 in, 0 out)
   3 Microsoft Sound Mapper - Output, MME (0 in, 2 out)
<  4 Speaker/Headphone (2- Realtek H, MME (0 in, 2 out)
   5 Primary Sound Capture Driver, Windows DirectSound (2 in, 0 out)
   6 Microphone (2- Realtek High Definition Audio), Windows DirectSound (2 in, 0 out)
   7 Stereo Mix (2- Realtek High Definition Audio), Windows DirectSound (2 in, 0 out)
   8 Primary Sound Driver, Windows DirectSound (0 in, 2 out)
   9 Speaker/Headphone (2- Realtek High Definition Audio), Windows DirectSound (0 in, 2 out)
  10 Speaker/Headphone (2- Realtek High Definition Audio), Windows WASAPI (0 in, 2 out)
  11 Microphone (2- Realtek High Definition Audio), Windows WASAPI (2 in, 0 out)
  12 Stereo Mix (2- Realtek High Definition Audio), Windows WASAPI (2 in, 0 out)
  13 Microphone (Realtek HD Audio Mic input), Windows WDM

In [25]:
SAMPLE_RATE = 16000  # 16 kHz
DURATION = 10

print('🎤 Recording for 5 seconds — speak now...')
recording = sd.rec(
    int(DURATION * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype='int16',
)
sd.wait()
print('Recording done')

# Save as WAV
out_path = r'C:\Users\USER\rag_course\12b_voice_agents\data\mic_test.wav'
sf.write(out_path, recording, SAMPLE_RATE)
print(f'💾 Saved: {out_path}')


🎤 Recording for 5 seconds — speak now...
Recording done
💾 Saved: C:\Users\USER\rag_course\12b_voice_agents\data\mic_test.wav


In [26]:
display(Audio(out_path))

In [27]:
segments, info = stt_model.transcribe(out_path, beam_size=5)

print(f'Language: {info.language} (probability {info.language_probability:.2f})')
print('Text:')
for seg in segments:
    print(f'  {seg.text}')

Language: en (probability 0.41)
Text:
   Hi everyone, today I was just finished with my demo for Wemma Bank of Hakaton.


In [40]:
import sounddevice as sd
import soundfile as sf
import numpy as np

SAMPLE_RATE = 16000
DURATION = 5

# Force device 1 (the Realtek Microphone)
recording = sd.rec(
    int(DURATION * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype='int16',
    device=1,
)

print('🎤 Recording for 5 seconds — speak now...')
sd.wait()

# Check volume immediately
peak = np.abs(recording).max()
print(f'✅ Done. Peak level: {peak} (out of 32767)')

if peak < 500:
    print('❌ Still silent — device is wrong or muted')
elif peak < 3000:
    print('⚠️  Quiet — try speaking louder or moving closer')
else:
    print('✅ Good signal')

out_path = r'C:\Users\USER\rag_course\12b_voice_agents\data\mic_test.wav'
sf.write(out_path, recording, SAMPLE_RATE)
print(f'💾 Saved: {out_path}')

🎤 Recording for 5 seconds — speak now...
✅ Done. Peak level: 31732 (out of 32767)
✅ Good signal
💾 Saved: C:\Users\USER\rag_course\12b_voice_agents\data\mic_test.wav


In [41]:
import sounddevice as sd
import soundfile as sf
import numpy as np

SAMPLE_RATE = 16000
DURATION = 5

# Force device 1 (the Realtek Microphone)
recording = sd.rec(
    int(DURATION * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype='int16',
    device=1,
)

print('🎤 Recording for 5 seconds — speak now...')
sd.wait()

# Check volume immediately
peak = np.abs(recording).max()
print(f'✅ Done. Peak level: {peak} (out of 32767)')

if peak < 500:
    print('❌ Still silent — device is wrong or muted')
elif peak < 3000:
    print('⚠️  Quiet — try speaking louder or moving closer')
else:
    print('✅ Good signal')

out_path = r'C:\Users\USER\rag_course\12b_voice_agents\data\mic_test.wav'
sf.write(out_path, recording, SAMPLE_RATE)
print(f'💾 Saved: {out_path}')

🎤 Recording for 5 seconds — speak now...
✅ Done. Peak level: 30958 (out of 32767)
✅ Good signal
💾 Saved: C:\Users\USER\rag_course\12b_voice_agents\data\mic_test.wav


In [42]:
display(Audio(out_path))

Transcribe

In [43]:
segments, info = stt_model.transcribe(out_path, beam_size=5)

print(f'Language: {info.language} ({info.language_probability:.2f})')
print('Text:')
for seg in segments:
    print(f'  {seg.text}')

Language: en (0.50)
Text:
   How did I trick us from the most I did this?


Chain the mic into VoicePipeline

In [44]:
class VoicePipeline:
    '''One voice turn: audio in → STT → RAG → TTS → audio out.'''

    # Store config once so every method can reuse it without reloading
    def __init__(self, language: str = 'English'):
        self.language = language
        print(f'✅ VoicePipeline created (language={self.language})')

    # Record from the microphone and save to a WAV file
    def record(self, seconds: int = 5, device: int = 1) -> str:
        import sounddevice as sd
        import soundfile as sf

        sample_rate = 16000
        print(f'🎤 Recording for {seconds} seconds — speak now...')

        audio = sd.rec(
            int(seconds * sample_rate),
            samplerate=sample_rate,
            channels=1,
            dtype='int16',
            device=device,
        )
        sd.wait()

        out_path = r'C:\Users\USER\rag_course\12b_voice_agents\data\mic_live.wav'
        sf.write(out_path, audio, sample_rate)
        print(f'✅ Saved: {out_path}')
        return out_path

    # Transcribe an audio file to text using local faster-whisper
    def listen(self, audio_path: str) -> str:
        segments, info = stt_model.transcribe(audio_path, beam_size=5)
        return ' '.join(seg.text.strip() for seg in segments)

    # Translate to English → retrieve → answer → translate back to user's language
    def think(self, question: str) -> str:
        question_en = translate_to_english(question, self.language)

        docs = retriever.invoke(question_en)
        context = '\n\n'.join(doc.page_content for doc in docs)
        answer_en = rag_chain.invoke({'context': context, 'question': question_en})

        return translate(answer_en, self.language)

    # Convert the answer text into spoken audio bytes with OpenAI TTS
    def speak(self, answer: str) -> bytes:
        response = client.audio.speech.create(
            model='tts-1',
            voice='alloy',
            input=answer,
        )
        return response.content

    # Chain listen → think → speak — the full one-turn voice flow
    def turn(self, audio_path: str) -> bytes:
        print('🎧 Listening...')
        question = self.listen(audio_path)
        print(f'   📝 {question}')

        print('🧠 Thinking...')
        answer = self.think(question)
        print(f'   💬 {answer}')

        print('🔊 Speaking...')
        audio_bytes = self.speak(answer)

        return audio_bytes

    # Full mic-to-speaker turn: record → listen → think → speak → play
    def conversation_turn(self, seconds: int = 5) -> None:
        from IPython.display import Audio, display

        # 1. Record from mic
        audio_path = self.record(seconds=seconds)

        # 2. Run the pipeline
        audio_bytes = self.turn(audio_path)

        # 3. Save and play the answer
        out_path = r'C:\Users\USER\rag_course\12b_voice_agents\data\answer_live.mp3'
        with open(out_path, 'wb') as f:
            f.write(audio_bytes)
        print(f'💾 Saved answer: {out_path}')

        display(Audio(out_path))


print('✅ VoicePipeline with conversation_turn() ready')

✅ VoicePipeline with conversation_turn() ready


In [46]:
vp = VoicePipeline(language='English')
vp.conversation_turn(seconds=5)

✅ VoicePipeline created (language=English)
🎤 Recording for 5 seconds — speak now...
✅ Saved: C:\Users\USER\rag_course\12b_voice_agents\data\mic_live.wav
🎧 Listening...
   📝 How can I treat cassava disease in Nigeria?
🧠 Thinking...
   💬 Use disease-free cuttings for planting, choose resistant cassava varieties, and remove and destroy infected plants immediately. Control whitefly vectors with appropriate insecticides and avoid planting near infected fields.
🔊 Speaking...
💾 Saved answer: C:\Users\USER\rag_course\12b_voice_agents\data\answer_live.mp3


In [48]:
vp = VoicePipeline(language='Yoruba')
vp.conversation_turn(seconds=5)

✅ VoicePipeline created (language=Yoruba)
🎤 Recording for 5 seconds — speak now...
✅ Saved: C:\Users\USER\rag_course\12b_voice_agents\data\mic_live.wav
🎧 Listening...
   📝 How can I treat cassava disease in Nigeria?
🧠 Thinking...
   💬 Lati tọju àrùn mosaic ẹgẹ́ ni Nàìjíríà, lo ẹ̀ka tí kò ní àrùn fún ìkó, yan irú ẹgẹ́ tí ó ní ìdàgbàsókè, kí o sì yọ́kúrò kí o sì pa àwọn ọgbin tí ó ní àrùn. Ṣakoso kòkòrò funfun pẹ̀lú àwọn inṣẹ́ktisáìd tó yẹ, kí o sì yago fún ìkó ẹgẹ́ nítorí àwọn pẹ́tẹ́lẹ̀ tí ó ní àrùn.
🔊 Speaking...
💾 Saved answer: C:\Users\USER\rag_course\12b_voice_agents\data\answer_live.mp3


In [55]:
vp = VoicePipeline(language='Hausa')
vp.conversation_turn(seconds=5)

✅ VoicePipeline created (language=Hausa)
🎤 Recording for 5 seconds — speak now...
✅ Saved: C:\Users\USER\rag_course\12b_voice_agents\data\mic_live.wav
🎧 Listening...
   📝 От кънът се дезисът афете казава и негерас систем.
🧠 Thinking...
   💬 Ban sani ba.
🔊 Speaking...
💾 Saved answer: C:\Users\USER\rag_course\12b_voice_agents\data\answer_live.mp3
